In [1]:
import duckdb

conn = duckdb.connect()
conn.execute("PRAGMA threads=8")
conn.execute("PRAGMA enable_object_cache=true")

market_path = "Data/data/polymarket/markets/markets_*.parquet"
trade_path = "Data/data/polymarket/trades/trades_*.parquet"

In [2]:
threshold = 0.5

market_query = f"""
CREATE TABLE markets
AS SELECT
    question,
    clob_token_ids ->> '$[0]' AS yes_token,
    clob_token_ids ->> '$[1]' AS no_token,
    CAST(outcome_prices ->> '$[0]' AS FLOAT) AS yes_price,
    CAST(outcome_prices ->> '$[1]' AS FLOAT) AS no_price
FROM '{market_path}';
CREATE INDEX idx_yes_token ON markets(yes_token);
CREATE INDEX idx_no_token ON markets(no_token);
"""

tokens_query = f"""
CREATE TABLE tokens
AS
SELECT
    yes_token AS token,
    CASE
        WHEN yes_price > {(100 - threshold) / 100.0} AND no_price < {threshold / 100.0} THEN TRUE
        WHEN yes_price < {threshold / 100.0} AND no_price > {(100 - threshold) / 100.0} THEN FALSE
        ELSE NULL
    END AS implied_outcome,
    CASE
        WHEN (yes_price > {(100 - threshold) / 100.0} AND no_price < {threshold / 100.0})
          OR (yes_price < {threshold / 100.0} AND no_price > {(100 - threshold) / 100.0}) THEN TRUE
        ELSE FALSE
    END AS resolved,
    1 = 1 as token_type
FROM markets

UNION ALL

SELECT
    no_token AS token,
    CASE
        WHEN yes_price > {(100 - threshold) / 100.0} AND no_price < {threshold / 100.0} THEN FALSE
        WHEN yes_price < {threshold / 100.0} AND no_price > {(100 - threshold) / 100.0} THEN TRUE
        ELSE NULL
    END AS implied_outcome,
    CASE
        WHEN (yes_price > {(100 - threshold) / 100.0} AND no_price < {threshold / 100.0})
          OR (yes_price < {threshold / 100.0} AND no_price > {(100 - threshold) / 100.0}) THEN TRUE
        ELSE FALSE
    END AS resolved,
    1 = 0 as token_type
FROM markets;
CREATE INDEX idx_token ON tokens(token);
"""

conn.execute("DROP TABLE IF EXISTS markets")
conn.execute(market_query)
conn.execute("DROP TABLE IF EXISTS tokens")
conn.execute(tokens_query)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [4]:
conn.execute("SELECT token_type, implied_outcome, count(*) FROM tokens GROUP BY token_type, implied_outcome").df()

,token_type,implied_outcome,count_star()
0,True,False,228869
1,False,<NA>,23938
2,False,False,156056
3,True,True,156056
4,False,True,228869
5,True,<NA>,23938


In [5]:
# print(conn.execute(f"select count(*) from '{trade_path}'").df())

In [6]:
print(conn.execute(f"select count(*) from '{market_path}'").df())

   count_star()
0        408863


In [7]:
query = f"""
SELECT * FROM '{trade_path}' LIMIT 1
"""

df = conn.execute(query).fetch_df()
# print each col and value
for col in df.columns:
    print(f"{col}: {df[col][0]}")

block_number: 40000176
transaction_hash: d04d37c8b4e36bfdc3469d0e69dd4f43579680cd7707153623236a8ba2a25eb9
log_index: 179
order_hash: d5552e9db0847fa6dc51bfaaf0d9957e8836d9c044ba711ef05fe898fbe2627e
maker: 0xd1acd3925D895DE9AEC98Ff95F3A30C5279d08d5
taker: 0xE58Ec10d493AC0916B0B1b0a802AAEfF3BAd43e7
maker_asset_id: 0
taker_asset_id: 108430748466715852616988667809810953079156970963086484559738136168402442879009
maker_amount: 73000000
taker_amount: 100000000
fee: 0
timestamp: <NA>
_fetched_at: 2026-01-29 15:48:12.728779
_contract: CTF Exchange


In [8]:
query = f"""
SELECT * FROM '{market_path}'
"""

df = conn.execute(query).fetch_df()
# print each col and value
for col in df.columns:
    print(f"{col}: {df[col][999]}")

id: 238944
condition_id: 0x10158e73d045f7ebc56840dd6e923c0e6c79192986f737db99f1d51982565514
question: Will Facebook be online at midnight ET, October 5th?
slug: will-facebook-be-online-at-midnight-october-5th
outcomes: ["Yes", "No"]
outcome_prices: ["0.9999998236377180802320302155002555", "0.0000001763622819197679697844997444570283"]
clob_token_ids: ["26417637768055845457225750224787802527285527876965833056341850820493311813897", "77760302591009336863018316565500592129254468620240993410073366710735525705114"]
volume: 10888.095548
liquidity: 438.929086
active: True
closed: True
end_date: 2021-10-04 20:00:00-04:00
created_at: 2021-10-04 15:59:43.177000-04:00
market_maker_address: 0xF82258820FD1b38ddcbB6ECEFD237141c76f2735
_fetched_at: 2026-02-03 22:24:37.376319


In [3]:
trades_by_maker = conn.execute(
    f"""
    SELECT
        maker,
        COUNT(*) AS trade_count
    FROM '{trade_path}'
    GROUP BY maker
    HAVING trade_count > 100
    ORDER BY trade_count DESC
    """
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [4]:
trades_by_maker

,maker,trade_count
0,0x6031B6eed1C97e853c6e0F03Ad3ce3529351F96d,8905579
1,0x7F69983eB28245Bba0D5083502a78744a8f66162,3503869
2,0xd44e29936409019F93993de8bd603EF6CB1BB15e,2980450
3,0xD218E474776403A330142299f7796E8BA32Eb5C9,2371230
4,0xca85f4b9e472B542e1DF039594eeAebb6d466bF2,2321043
...,...,...
235187,0xaC0FAbA11dF89e57901e40DDA780e2F5c8ebBdFc,101
235188,0xe2f3E4e39e36A7F110E0962c719C9F1b6D1a8CBD,101
235189,0xDf4128d5E444261F8cdcAC4586abB934e5b846aa,101
235190,0x53E88b34292352c4EeD7Dfa6B6dF083A01C677A3,101


In [5]:
from functools import lru_cache

@lru_cache(maxsize=1000)
def getExactAddress(targetAddress):
    query = f"""
    SELECT maker FROM '{trade_path}' WHERE UPPER(maker) = UPPER('{targetAddress}') LIMIT 1
    """
    df = conn.execute(query).fetch_df()
    return df['maker'][0] if len(df) > 0 else None

In [6]:
getExactAddress("0x3ee2bbb93614edeea3988476e9125224b707e091")

'0x3EE2bbb93614EdeEA3988476e9125224b707E091'

In [7]:
target_address = "0x3ee2bbb93614edeea3988476e9125224b707e091"
exact_address = getExactAddress(target_address)

# target_address = target_address.upper()
query = f"""
SELECT
    COUNT(*)
FROM '{trade_path}'
WHERE
    maker = '{exact_address}'
    or taker = '{exact_address}'
"""

df = conn.execute(query).fetch_df()
# print each col and value
for col in df.columns:
    print(f"{col}: {df[col][0]}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

count_star(): 576


In [8]:
target_address = "0x3ee2bbb93614edeea3988476e9125224b707e091"
exact_address = getExactAddress(target_address)

query = f"""
CREATE TABLE user_trades AS
SELECT 
    maker,
    taker,
    maker_amount,
    taker_amount,
    block_number,
    taker_asset_id,
    maker_asset_id,
FROM '{trade_path}'
WHERE
    maker = '{exact_address}'
    or taker = '{exact_address}'
"""

conn.execute(query)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [9]:
query = f"""
WITH rich_user_trades AS (
    SELECT
        *,
        CASE
            WHEN maker_asset_id = '0' THEN taker_asset_id
            ELSE maker_asset_id
        END AS token,
        CASE
            WHEN maker_asset_id = '0' THEN maker = '{exact_address}'
            ELSE taker = '{exact_address}'
        END AS buyer
    FROM user_trades
)
SELECT
    rich_user_trades.*,
    question,
    tokens.implied_outcome
FROM rich_user_trades
JOIN
    markets on rich_user_trades.token IN (markets.yes_token, markets.no_token)
JOIN
    tokens on rich_user_trades.token = tokens.token
ORDER BY block_number DESC
"""

df = conn.execute(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [10]:
def getPriceAndQuantity(trade):
    if trade[1]['maker_asset_id'] == '0': # maker asset is usdc
        dollar_cost = trade[1]['maker_amount'] / 1e6
        quantity = trade[1]['taker_amount'] / 1e6
        price = dollar_cost / quantity
        return price, quantity
    else:
        dollar_cost = trade[1]['taker_amount'] / 1e6
        quantity = trade[1]['maker_amount'] / 1e6
        price = dollar_cost / quantity
        return price, quantity

def getProfit(trade):
    price, quantity = getPriceAndQuantity(trade)
    base = price * quantity
    implied_outcome = trade[1]['implied_outcome']
    correct = False if type(implied_outcome).__name__ == 'NAType' else bool(implied_outcome)
    if trade[1]['buyer']:
        return (quantity if correct else 0) - base
    else:
        return base - (quantity if correct else 0)

totalProfit = 0
for trade in df.iterrows():
    price, quantity = getPriceAndQuantity(trade)
    totalProfit += getProfit(trade)
    print(trade[1]['question'], "Buy" if trade[1]['buyer'] else "Sell", f"{quantity:.2f}", f"{price:.2f}", totalProfit)

print(totalProfit)

Will Courtney Najera lose CA-30 primary? Buy 47.98 0.01 -0.47979
Will Courtney Najera lose CA-30 primary? Buy 202.02 0.01 -2.4999919999999998
Will Ben Armstrong (Bitboy) get KO'd? Buy 2857.14 0.93 197.49980800000006
Will Ben Armstrong (Bitboy) get KO'd? Sell 2857.14 0.07 397.4996080000001
Will Ben Armstrong (Bitboy) get KO'd? Buy 2857.14 0.07 197.4996090000001
Will Ben Armstrong (Bitboy) get KO'd? Sell 1857.14 0.93 67.49961000000022
Will Ben Armstrong (Bitboy) get KO'd? Sell 1000.00 0.93 -2.5003899999997827
Will Solana Network go down again in February? Sell 200.00 0.96 -10.300389999999794
Will Solana Network go down again in February? Sell 200.00 0.96 -18.100389999999805
Will Ben Armstrong (Bitboy) beat More Light? Buy 250.00 0.30 156.8996100000002
Will More Light beat Ben Armstrong? Buy 250.00 0.60 256.89961000000017
Ethereum all time high in 2024? Buy 217.39 0.46 156.89961100000016
Ethereum all time high in 2024? Sell 169.57 0.54 78.89681100000017
Ethereum all time high in 2024? Buy

In [ ]:
query = f"""
WITH resolved_trades AS (
    SELECT
        maker,
        taker,
        maker_asset_id,
        CASE
            WHEN maker_asset_id = '0' THEN taker_asset_id
            ELSE maker_asset_id
        END AS token
    FROM '{trade_path}'
),
user_trade_outcomes AS (
    SELECT
        resolved_trades.maker AS user_address,
        tokens.implied_outcome = TRUE AS won
    FROM resolved_trades
    JOIN tokens ON resolved_trades.token = tokens.token
    WHERE tokens.resolved = TRUE AND resolved_trades.maker_asset_id = '0'
    UNION ALL
    SELECT
        resolved_trades.taker AS user_address,
        tokens.implied_outcome = TRUE AS won
    FROM resolved_trades
    JOIN tokens ON resolved_trades.token = tokens.token
    WHERE tokens.resolved = TRUE AND resolved_trades.maker_asset_id <> '0'
),
user_win_rates AS (
    SELECT
        user_address,
        COUNT(*) AS trade_count,
        SUM(CASE WHEN won THEN 1 ELSE 0 END) AS wins,
        AVG(CASE WHEN won THEN 1.0 ELSE 0.0 END) AS win_rate
    FROM user_trade_outcomes
    GROUP BY user_address
    HAVING COUNT(*) > 50
    ORDER BY win_rate DESC, trade_count DESC
    LIMIT 100
)
SELECT *
FROM user_win_rates
"""

df = conn.execute(query).df()
df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

RuntimeError: Query interrupted